In [ ]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.preprocessing import StandardScaler
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

# Load the dataset
df = pd.read_csv('/content/EVSE-B-PowerCombined.csv')

# Drop irrelevant columns if needed
df = df[['time', 'shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW', 'State', 'Attack', 'Label']]

# Convert labels to binary (0 for normal, 1 for attack)
df['Label'] = df['Label'].apply(lambda x: 1 if x == 'attack' else 0)

# Standardize the features
scaler = StandardScaler()
features = scaler.fit_transform(df[['shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW']])

# Create edge index (assuming each row is a node connected to the next for simplicity)
edge_index = torch.tensor([[i, i + 1] for i in range(len(df) - 1)], dtype=torch.long).t()

# Convert to PyTorch Geometric data format
x = torch.tensor(features, dtype=torch.float)
y = torch.tensor(df['Label'].values, dtype=torch.long)
data = Data(x=x, edge_index=edge_index, y=y)

# Train-validation-test split
train_data, temp_data = train_test_split(list(range(len(df))), test_size=0.4, stratify=y)
val_data, test_data = train_test_split(temp_data, test_size=0.5, stratify=y[temp_data])

train_mask = torch.tensor([i in train_data for i in range(len(df))], dtype=torch.bool)
val_mask = torch.tensor([i in val_data for i in range(len(df))], dtype=torch.bool)
test_mask = torch.tensor([i in test_data for i in range(len(df))], dtype=torch.bool)

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

train_loader = DataLoader([data], batch_size=1, shuffle=True)
val_loader = DataLoader([data], batch_size=1)
test_loader = DataLoader([data], batch_size=1)

# Define the GRNN model
class EVChargingGRNN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(EVChargingGRNN, self).__init__()
        self.gat = GATConv(input_dim, hidden_dim)
        self.rnn = torch.nn.RNN(hidden_dim, hidden_dim, batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, output_dim)
        self.dropout = torch.nn.Dropout(0.3)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        # Graph layer
        x = F.relu(self.gat(x, edge_index))
        x = self.dropout(x)

        # Prepare data for RNN (seq_len, batch_size, features)
        x = x.unsqueeze(1)  # Add a batch dimension
        x, _ = self.rnn(x)  # Get all RNN outputs
        x = x.squeeze(1)    # Remove the batch dimension

        # Fully connected layer for classification
        x = self.fc(x)
        return torch.sigmoid(x)

# Instantiate the model
model = EVChargingGRNN(input_dim=x.shape[1], hidden_dim=16, output_dim=1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = torch.nn.BCELoss()

# Training function
def train():
    model.train()
    total_loss = 0
    for data in train_loader:
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output[data.train_mask].squeeze(), data.y[data.train_mask].float())
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

# Evaluation function
def evaluate(loader, mask_type):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for data in loader:
            output = model(data)
            mask = getattr(data, f"{mask_type}_mask")
            preds = (output[mask] > 0.5).float()
            all_preds.append(preds.cpu())
            all_labels.append(data.y[mask].cpu())
            all_probs.append(output[mask].cpu())

    # Concatenate predictions, labels, and probabilities
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    all_probs = torch.cat(all_probs).numpy()

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    roc_auc = None
    if len(set(all_labels)) > 1:
        roc_auc = roc_auc_score(all_labels, all_probs)

    return accuracy, precision, f1, roc_auc

# Training loop with validation
for epoch in range(50):
    loss = train()
    val_accuracy, val_precision, val_f1, val_auc = evaluate(val_loader, 'val')
    print(f'Epoch {epoch+1}, Loss: {loss:.4f}, Val Accuracy: {val_accuracy:.4f}, Val Precision: {val_precision:.4f}, Val F1: {val_f1:.4f}, Val AUC: {val_auc:.4f}')

# Test evaluation
test_accuracy, test_precision, test_f1, test_roc_auc = evaluate(test_loader, 'test')
print(f'Test Accuracy: {test_accuracy:.4f}')
print(f'Test Precision: {test_precision:.4f}')
print(f'Test F1 Score: {test_f1:.4f}')
print(f'Test ROC-AUC: {test_roc_auc:.4f}')


Epoch 1, Loss: 0.7850, Val Accuracy: 0.3198, Val Precision: 0.9759, Val F1: 0.3706, Val AUC: 0.5244
Epoch 2, Loss: 0.7144, Val Accuracy: 0.8758, Val Precision: 0.8766, Val F1: 0.9337, Val AUC: 0.5864
Epoch 3, Loss: 0.6528, Val Accuracy: 0.8755, Val Precision: 0.8755, Val F1: 0.9336, Val AUC: 0.6511
Epoch 4, Loss: 0.5996, Val Accuracy: 0.8755, Val Precision: 0.8755, Val F1: 0.9336, Val AUC: 0.6637
Epoch 5, Loss: 0.5545, Val Accuracy: 0.8755, Val Precision: 0.8755, Val F1: 0.9336, Val AUC: 0.6683
Epoch 6, Loss: 0.5160, Val Accuracy: 0.8755, Val Precision: 0.8755, Val F1: 0.9336, Val AUC: 0.6713
Epoch 7, Loss: 0.4830, Val Accuracy: 0.8755, Val Precision: 0.8755, Val F1: 0.9336, Val AUC: 0.6734
Epoch 8, Loss: 0.4545, Val Accuracy: 0.8755, Val Precision: 0.8755, Val F1: 0.9336, Val AUC: 0.6750
Epoch 9, Loss: 0.4310, Val Accuracy: 0.8755, Val Precision: 0.8755, Val F1: 0.9336, Val AUC: 0.6764
Epoch 10, Loss: 0.4104, Val Accuracy: 0.8755, Val Precision: 0.8755, Val F1: 0.9336, Val AUC: 0.6775

In [ ]:
import pandas as pd
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.preprocessing import StandardScaler
import torch.nn.functional as F
from torch_geometric.nn import GATConv, BatchNorm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

# Load the dataset
df = pd.read_csv('/content/EVSE-B-PowerCombined.csv')

# Drop irrelevant columns if needed
df = df[['time', 'shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW', 'State', 'Attack', 'Label']]

# Convert labels to binary (0 for normal, 1 for attack)
df['Label'] = df['Label'].apply(lambda x: 1 if x == 'attack' else 0)

# Standardize the features
scaler = StandardScaler()
features = scaler.fit_transform(df[['shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW']])

# Create edge index (assuming each row is a node connected to the next for simplicity)
edge_index = torch.tensor([[i, i + 1] for i in range(len(df) - 1)], dtype=torch.long).t()

# Convert to PyTorch Geometric data format
x = torch.tensor(features, dtype=torch.float)
y = torch.tensor(df['Label'].values, dtype=torch.long)
data = Data(x=x, edge_index=edge_index, y=y)

# Train-validation-test split
train_data, temp_data = train_test_split(list(range(len(df))), test_size=0.4, stratify=y)
val_data, test_data = train_test_split(temp_data, test_size=0.5, stratify=y[temp_data])

train_mask = torch.tensor([i in train_data for i in range(len(df))], dtype=torch.bool)
val_mask = torch.tensor([i in val_data for i in range(len(df))], dtype=torch.bool)
test_mask = torch.tensor([i in test_data for i in range(len(df))], dtype=torch.bool)

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

train_loader = DataLoader([data], batch_size=1, shuffle=True)
val_loader = DataLoader([data], batch_size=1)
test_loader = DataLoader([data], batch_size=1)

# Define the improved GRNN model
class EVChargingGRNN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(EVChargingGRNN, self).__init__()
        self.gat1 = GATConv(input_dim, hidden_dim)
        self.gat2 = GATConv(hidden_dim, hidden_dim)
        self.batch_norm = BatchNorm(hidden_dim)
        self.rnn = torch.nn.RNN(hidden_dim, hidden_dim, batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, output_dim)
        self.dropout = torch.nn.Dropout(0.4)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        # Graph layers
        x = F.relu(self.gat1(x, edge_index))
        x = self.batch_norm(x)
        x = F.relu(self.gat2(x, edge_index))
        x = self.dropout(x)

        # Prepare data for RNN (seq_len, batch_size, features)
        x = x.unsqueeze(1)  # Add a batch dimension
        x, _ = self.rnn(x)  # Get all RNN outputs
        x = x.squeeze(1)    # Remove the batch dimension

        # Fully connected layer for classification
        x = self.fc(x)
        return torch.sigmoid(x)

# Instantiate the model
model = EVChargingGRNN(input_dim=x.shape[1], hidden_dim=32, output_dim=1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.7)
criterion = torch.nn.BCELoss()

# Training function
def train():
    model.train()
    total_loss = 0
    for data in train_loader:
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output[data.train_mask].squeeze(), data.y[data.train_mask].float())
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    return total_loss / len(train_loader)

# Evaluation function
def evaluate(loader, mask_type):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for data in loader:
            output = model(data)
            mask = getattr(data, f"{mask_type}_mask")
            preds = (output[mask] > 0.5).float()
            all_preds.append(preds.cpu())
            all_labels.append(data.y[mask].cpu())
            all_probs.append(output[mask].cpu())

    # Concatenate predictions, labels, and probabilities
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    all_probs = torch.cat(all_probs).numpy()

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    roc_auc = None
    if len(set(all_labels)) > 1:
        roc_auc = roc_auc_score(all_labels, all_probs)

    return accuracy, precision, recall, f1, roc_auc

# Training loop with validation
best_val_f1 = 0
for epoch in range(100):  # Increased epochs
    loss = train()
    val_accuracy, val_precision, val_recall, val_f1, val_auc = evaluate(val_loader, 'val')
    print(f'Epoch {epoch+1}, Loss: {loss:.4f}, Val Accuracy: {val_accuracy:.4f}, Val Precision: {val_precision:.4f}, Val Recall: {val_recall:.4f}, Val F1: {val_f1:.4f}, Val AUC: {val_auc:.4f}')

    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'best_model.pth')

# Load the best model and evaluate on test set
model.load_state_dict(torch.load('best_model.pth'))
test_accuracy, test_precision, test_recall, test_f1, test_roc_auc = evaluate(test_loader, 'test')
print(f'Test Accuracy: {test_accuracy:.4f}')
print(f'Test Precision: {test_precision:.4f}')
print(f'Test Recall: {test_recall:.4f}')
print(f'Test F1 Score: {test_f1:.4f}')
print(f'Test ROC-AUC: {test_roc_auc:.4f}')


Epoch 1, Loss: 0.6230, Val Accuracy: 0.8754, Val Precision: 0.8754, Val Recall: 1.0000, Val F1: 0.9336, Val AUC: 0.6678
Epoch 2, Loss: 0.5549, Val Accuracy: 0.8754, Val Precision: 0.8754, Val Recall: 1.0000, Val F1: 0.9336, Val AUC: 0.6752
Epoch 3, Loss: 0.4997, Val Accuracy: 0.8754, Val Precision: 0.8754, Val Recall: 1.0000, Val F1: 0.9336, Val AUC: 0.6779
Epoch 4, Loss: 0.4534, Val Accuracy: 0.8754, Val Precision: 0.8754, Val Recall: 1.0000, Val F1: 0.9336, Val AUC: 0.6798
Epoch 5, Loss: 0.4163, Val Accuracy: 0.8754, Val Precision: 0.8754, Val Recall: 1.0000, Val F1: 0.9336, Val AUC: 0.6786
Epoch 6, Loss: 0.3882, Val Accuracy: 0.8754, Val Precision: 0.8754, Val Recall: 1.0000, Val F1: 0.9336, Val AUC: 0.6758
Epoch 7, Loss: 0.3692, Val Accuracy: 0.8754, Val Precision: 0.8754, Val Recall: 1.0000, Val F1: 0.9336, Val AUC: 0.6741
Epoch 8, Loss: 0.3579, Val Accuracy: 0.8754, Val Precision: 0.8754, Val Recall: 1.0000, Val F1: 0.9336, Val AUC: 0.6738
Epoch 9, Loss: 0.3525, Val Accuracy: 0.8

<ipython-input-4-05784a6146cb>:140: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model.pth'))


Test Accuracy: 0.8754
Test Precision: 0.8756
Test Recall: 0.9998
Test F1 Score: 0.9336
Test ROC-AUC: 0.8199
